# ✦ LILY WAN 2.2 — Dual-T4 Studio — FIXED v2

Set **Internet ON** and **GPU T4 x2**, then run the one code cell. This version avoids Kaggle core-package upgrades and completely skips the optional RIFE clone during startup so the UI can actually launch.


In [ ]:
import json, urllib.request, importlib.util, sys, subprocess

print('✦ Loading Lily Wan 2.2 Studio — safe startup v2...')

SOURCE_URL = 'https://raw.githubusercontent.com/benruiz1024-ops/hi/789b857e85738efdaec591f04de11bf76befe20d/LILY_WAN22_DUAL_T4_STUDIO.ipynb'
with urllib.request.urlopen(SOURCE_URL, timeout=60) as r:
    original = json.loads(r.read().decode('utf-8'))

code_cells = [c for c in original['cells'] if c.get('cell_type') == 'code']
if not code_cells:
    raise RuntimeError('Could not locate the studio code cell.')
code = ''.join(code_cells[0]['source'])

# 1) Never blanket-upgrade Kaggle's existing requests/pandas/jupyter stack.
bad_core = '''# Core helpers/UI.
pip_install("gradio>=5.20,<6", "huggingface_hub>=0.29", "requests>=2.32", "gdown>=5.2", "scikit-video")'''
safe_core = '''# Core helpers/UI — SAFE KAGGLE INSTALL.
import importlib.util
_missing = []
for _mod, _pkg in [("gradio", "gradio"), ("huggingface_hub", "huggingface_hub"), ("gdown", "gdown")]:
    if importlib.util.find_spec(_mod) is None:
        _missing.append(_pkg)
if _missing:
    run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *_missing], check=False)
else:
    print("✓ Kaggle core packages already present — no upgrades needed")'''
if bad_core not in code:
    raise RuntimeError('Safety patch target 1 was not found.')
code = code.replace(bad_core, safe_core, 1)

# 2) ComfyUI requirements warnings must not abort startup.
old_req = 'run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=COMFY)'
new_req = 'run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "-r", "requirements.txt"], cwd=COMFY, check=False)'
if old_req in code:
    code = code.replace(old_req, new_req, 1)

# 3) Defuse the helper too.
old_helper = '''def pip_install(*pkgs):
    run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *pkgs])'''
new_helper = '''def pip_install(*pkgs):
    run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *pkgs], check=False)'''
if old_helper in code:
    code = code.replace(old_helper, new_helper, 1)

# 4) IMPORTANT: skip Practical-RIFE entirely during startup.
# The previous version could hang forever at 'Cloning into Practical-RIFE'.
rife_start = '# -------------------------\n# 4) Optional RIFE on GPU 1'
rife_end = '# -------------------------\n# 5) Start ComfyUI on GPU 0'
a = code.find(rife_start)
b = code.find(rife_end)
if a == -1 or b == -1 or b <= a:
    raise RuntimeError('Could not locate the RIFE startup block to disable it safely.')
no_rife = '''# -------------------------
# 4) RIFE skipped for reliable Kaggle startup
# -------------------------
RIFE_READY = False
print("✓ Skipping optional RIFE clone — using ffmpeg interpolation fallback")

'''
code = code[:a] + no_rife + code[b:]

print('✓ Dependency safety patch applied')
print('✓ RIFE startup clone disabled')
print('✓ Starting Wan 2.2 setup...\n')

exec(compile(code, 'LILY_WAN22_DUAL_T4_STUDIO_SAFE_V2', 'exec'), globals(), globals())
